# Uncertainty-guided self-review

A LangGraph agent uses SAPLMA or UQ Head to score an archived draft,
reviews the most uncertain lines, and proposes a revision. Plain review
uses the same prompt and
generation budget with the uncertainty hints omitted.
[Agent pipeline diagram](../RESULTS.md#agent-pipeline)

Install the [root requirements](../requirements.txt) in the [project environment](README).
In VS Code, select the project interpreter at `.venv/Scripts/python.exe`.

For the Python checks, start Docker and run `docker pull python:3.11-slim`.
Set `check_python = False` to run the reference comparison alone.

In [ ]:
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS
from src.evaluation import TraceStore, run_benchmark

torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)
cases_per_benchmark = 4


Each method fits on that benchmark's original training split. Every
condition starts with the same held-out draft and gets one review round.
The agent sees the task, draft, and up to two ranked lines. The reference
solution and Python assertions are used by the evaluator.

Python cases are sampled from tasks whose canonical solution passes the archived assertions. The run uses a fixed seed.

In [ ]:
rows = []
for benchmark in DATASETS:
    rows.extend(
        run_benchmark(
            store,
            benchmark,
            limit=cases_per_benchmark,
            check_python=True,
        )
    )
    print(f"Finished {benchmark}")
results = pd.DataFrame(rows)
for benchmark, group in results.groupby("benchmark", sort=False):
    print(benchmark, ", ".join(group["task"].unique()))

mbpp:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Finished mbpp


Finished humaneval


Finished humaneval_java


Finished defects4j
mbpp 569, 391, 268, 723
humaneval 129, 127, 64, 115
humaneval_java 114, 36, 24, 139
defects4j Closure_52, Compress_30, Closure_35, JacksonDatabind_76


### Held-out results

Similarity compares code tokens with the archived canonical solution.
Exact matches use the same token representation. Python pass counts use
the archived assertions. Java results use the reference comparison.

In [3]:
plain = results.loc[results["method"] == "Plain review"]
baseline = plain.set_index(["benchmark", "task"])["similarity"]
results = results.join(baseline.rename("plain_similarity"), on=["benchmark", "task"])
results["delta_vs_plain"] = results["similarity"] - results["plain_similarity"]

grouped = results.groupby(["benchmark", "method"], sort=False)
summary = grouped.agg(
    tasks=("task", "size"),
    similarity=("similarity", "mean"),
    delta_vs_plain=("delta_vs_plain", "mean"),
    exact=("exact", "sum"),
    valid_replies=("accepted", "sum"),
)
summary["python_passes"] = grouped["passed"].sum(min_count=1).astype("Int64")
display(summary.round(4))

tasks  similarity  delta_vs_plain  exact  \
benchmark      method                                                     
mbpp           Archived draft      4      0.6357         -0.0913      0   
               Plain review        4      0.7269          0.0000      0   
               UQ Head             4      0.5893         -0.1377      0   
               SAPLMA              4      0.5893         -0.1377      0   
humaneval      Archived draft      4      0.4115         -0.1376      0   
               Plain review        4      0.5491          0.0000      0   
               UQ Head             4      0.5258         -0.0234      0   
               SAPLMA              4      0.5313         -0.0178      0   
humaneval_java Archived draft      4      0.5843          0.0525      0   
               Plain review        4      0.5318          0.0000      0   
               UQ Head             4      0.6399          0.1081      0   
               SAPLMA              4      0.5831          0.0512      0   
defects4j      Archived draft      4      0.2808          0.0355      0   
               Plain review        4      0.2453          0.0000      0   
               UQ Head             4      0.3129          0.0675      0   
               SAPLMA              4      0.3107          0.0654      0   

                               valid_replies  python_passes  
benchmark      method                                        
mbpp           Archived draft              4              0  
               Plain review                4              1  
               UQ Head                     4              2  
               SAPLMA                      4              2  
humaneval      Archived draft              4              0  
               Plain review                3              0  
               UQ Head                     3              1  
               SAPLMA                      3              0  
humaneval_java Archived draft              4           <NA>  
               Plain review                4           <NA>  
               UQ Head                     4           <NA>  
               SAPLMA                      4           <NA>  
defects4j      Archived draft              4           <NA>  
               Plain review                4           <NA>  
               UQ Head                     4           <NA>  
               SAPLMA                      3           <NA>

In [4]:
example = results.loc[results["method"] == "SAPLMA"].iloc[0]
print(f"{example['benchmark']} task {example['task']}")
print("Hints passed to the agent")
for line, excerpt in example["hints"]:
    print(f"{line}: {excerpt}")
print(example["review"])

mbpp task 569
Hints passed to the agent
3: return sorted(list1, key=lambda x: sorted(x))
1: def sort_sublists(list1):
```python
def sort_sublists(list1):
    return sorted(list1, key=lambda x: tuple(sorted(x)))
```

Review: The current draft is almost correct, but it needs a slight modification to ensure that the sorting is stable and the sublists are sorted in-place.

The issue with the current draft is that it sorts the sublists using the `sorted` function, which returns a new sorted list and leaves the original list unchanged. This can be inefficient if the sublists are large. To fix this, we can use the `sorted` function with the `key` argument set to a lambda function that returns a tuple of the sorted sublist. This ensures that the sorting is stable and the sublists are sorted in-place.

Alternatively, you could use the `sorted` function with the `key` argument set to a lambda function that returns the sorted sublist itself, like this:


In [9]:
print(*example)

mbpp 569 SAPLMA 0.9333333333333333 False True True [(3, 'return sorted(list1, key=lambda x: sorted(x))'), (1, 'def sort_sublists(list1):')] ```python
def sort_sublists(list1):
    return sorted(list1, key=lambda x: tuple(sorted(x)))
```

Review: The current draft is almost correct, but it needs a slight modification to ensure that the sorting is stable and the sublists are sorted in-place.

The issue with the current draft is that it sorts the sublists using the `sorted` function, which returns a new sorted list and leaves the original list unchanged. This can be inefficient if the sublists are large. To fix this, we can use the `sorted` function with the `key` argument set to a lambda function that returns a tuple of the sorted sublist. This ensures that the sorting is stable and the sublists are sorted in-place.

Alternatively, you could use the `sorted` function with the `key` argument set to a lambda function that returns the sorted sublist itself, like this: 0.9333333333333333 0.0

## Candidate selection

One model call proposes up to three distinct, complete code blocks.
Both estimators rank the same candidates. Each candidate is replayed
under the archived task prompt, and its score is the maximum predicted
defect score over code tokens. The agent keeps the lowest-scoring
candidate, with ties resolved in generation order. The draft is the
fallback when there are no usable candidates.

The two approaches start from the same held-out tasks. Line-guided
review allows 256 new tokens. Candidate selection allows up to 768
new tokens in one generation call. The evaluator scores only the
final answer returned by each approach.

In [10]:
from src.agent_loop import create_agent_candidates, extract_code

original = {"task": "Return the intended value.", "draft": "original"}
blocks = ["```python\nreturn " + str(i) + "\n```" for i in range(4)]
generated = "\n".join([blocks[0], blocks[0], *blocks[1:]])
scores = {"return 0": 0.8, "return 1": 0.2, "return 2": 0.2}
agent = create_agent_candidates(
    lambda prompt: generated,
    lambda reply: scores[extract_code(reply).strip()],
)
outcome = agent.invoke(original)
assert outcome["variants"] == blocks[:3]
assert outcome["uncertainties"] == [0.8, 0.2, 0.2]
assert outcome["selected"] == 1 and outcome["draft"] == blocks[1]

invalid = create_agent_candidates(lambda prompt: "```python\nreturn 0", lambda reply: float("nan"))
fallback = invalid.invoke(original)
assert fallback["selected"] is None and fallback["draft"] == original["draft"]
rejected = create_agent_candidates(lambda prompt: blocks[0], lambda reply: float("nan"))
assert rejected.invoke(original)["draft"] == original["draft"]
print("Candidate selection smoke checks passed.")

Candidate selection smoke checks passed.


In [11]:
from src.evaluation import run_benchmark

candidate_rows = []
for benchmark in DATASETS:
    candidate_rows.extend(
        run_benchmark(
            store,
            benchmark,
            limit=cases_per_benchmark,
            check_python=True,
            max_candidates=3,
        )
    )
    print(f"Finished candidate selection on {benchmark}")
candidate_results = pd.DataFrame(candidate_rows)

Finished candidate selection on mbpp


Finished candidate selection on humaneval


Finished candidate selection on humaneval_java


Finished candidate selection on defects4j


In [12]:
review = results.loc[
    results["method"].isin(candidate_results["method"].unique())
]
keys = ["benchmark", "task", "method"]
candidate_index = candidate_results.set_index(keys).index
review_index = review.set_index(keys).index
assert candidate_index.is_unique and review_index.is_unique
assert candidate_index.symmetric_difference(review_index).empty

paired = pd.concat([
    review.assign(approach="Line-guided review"),
    candidate_results.assign(approach="Candidate selection"),
], ignore_index=True)
groups = paired.groupby(["benchmark", "method", "approach"], sort=False)
comparison = groups.agg(
    tasks=("task", "size"),
    similarity=("similarity", "mean"),
    exact=("exact", "sum"),
    valid_replies=("accepted", "sum"),
    mean_candidates=("variants", "mean"),
)
comparison["python_passes"] = groups["passed"].sum(min_count=1).astype("Int64")
display(comparison.round(4))

tasks  similarity  exact  \
benchmark      method  approach                                        
mbpp           UQ Head Line-guided review       4      0.5893      0   
               SAPLMA  Line-guided review       4      0.5893      0   
humaneval      UQ Head Line-guided review       4      0.5258      0   
               SAPLMA  Line-guided review       4      0.5313      0   
humaneval_java UQ Head Line-guided review       4      0.6399      0   
               SAPLMA  Line-guided review       4      0.5831      0   
defects4j      UQ Head Line-guided review       4      0.3129      0   
               SAPLMA  Line-guided review       4      0.3107      0   
mbpp           UQ Head Candidate selection      4      0.5889      0   
               SAPLMA  Candidate selection      4      0.6282      0   
humaneval      UQ Head Candidate selection      4      0.5098      0   
               SAPLMA  Candidate selection      4      0.5888      0   
humaneval_java UQ Head Candidate selection      4      0.4923      0   
               SAPLMA  Candidate selection      4      0.4923      0   
defects4j      UQ Head Candidate selection      4      0.1538      0   
               SAPLMA  Candidate selection      4      0.1538      0   

                                            valid_replies  mean_candidates  \
benchmark      method  approach                                              
mbpp           UQ Head Line-guided review               4              NaN   
               SAPLMA  Line-guided review               4              NaN   
humaneval      UQ Head Line-guided review               3              NaN   
               SAPLMA  Line-guided review               3              NaN   
humaneval_java UQ Head Line-guided review               4              NaN   
               SAPLMA  Line-guided review               4              NaN   
defects4j      UQ Head Line-guided review               4              NaN   
               SAPLMA  Line-guided review               3              NaN   
mbpp           UQ Head Candidate selection              4             3.00   
               SAPLMA  Candidate selection              4             3.00   
humaneval      UQ Head Candidate selection              4             2.75   
               SAPLMA  Candidate selection              4             2.75   
humaneval_java UQ Head Candidate selection              4             2.00   
               SAPLMA  Candidate selection              4             2.00   
defects4j      UQ Head Candidate selection              4             1.50   
               SAPLMA  Candidate selection              4             1.50   

                                            python_passes  
benchmark      method  approach                            
mbpp           UQ Head Line-guided review               2  
               SAPLMA  Line-guided review               2  
humaneval      UQ Head Line-guided review               1  
               SAPLMA  Line-guided review               0  
humaneval_java UQ Head Line-guided review            <NA>  
               SAPLMA  Line-guided review            <NA>  
defects4j      UQ Head Line-guided review            <NA>  
               SAPLMA  Line-guided review            <NA>  
mbpp           UQ Head Candidate selection              0  
               SAPLMA  Candidate selection              1  
humaneval      UQ Head Candidate selection              0  
               SAPLMA  Candidate selection              0  
humaneval_java UQ Head Candidate selection           <NA>  
               SAPLMA  Candidate selection           <NA>  
defects4j      UQ Head Candidate selection           <NA>  
               SAPLMA  Candidate selection           <NA>

In [13]:
first_task = candidate_results.iloc[0]["task"]
first_benchmark = candidate_results.iloc[0]["benchmark"]
scores = candidate_results.loc[
    (candidate_results["benchmark"] == first_benchmark)
    & (candidate_results["task"] == first_task),
    ["method", "uncertainties", "selected_candidate"],
].copy()
scores["selected_candidate"] = (scores["selected_candidate"] + 1).astype("Int64")
print(f"Candidate scoring example for {first_benchmark} task {first_task}")
display(scores.set_index("method"))

Candidate scoring example for mbpp task 569


,uncertainties,selected_candidate
method,,
UQ Head,"[0.28881940245628357, 0.2887659966945648, 0.28...",2
SAPLMA,"[0.03702695667743683, 0.0485282726585865, 0.03...",3
